# Clinical Note Bias Assessment Tool

This notebook analyzes clinical notes for potentially biased or stigmatizing language using an LLM-based approach.

## Requirements

**Files needed:**
1. This notebook (`bias_assessment.ipynb`)
2. The prompt file (`bias_detection_prompt.py`) - must be in the same directory
3. Your input CSV file with a `note_text` column containing clinical notes

**Environment setup:**
1. Create a `.env` file in the same directory with your Azure OpenAI credentials:
   ```
   AZURE_OPENAI_ENDPOINT=https://your-endpoint.openai.azure.com/
   AZURE_DEPLOYMENT=your-deployment-name
   AZURE_SUBSCRIPTION_KEY=your-api-key
   AZURE_API_VERSION=2024-02-15-preview
   ```

2. Install required packages:
   ```bash
   pip install pandas openai python-dotenv
   ```

## Output

The tool generates a CSV file with two additional columns:
- **Possible_Biased_Terms**: Phrases that may reflect bias but are context-dependent
- **Likely_Biased_Terms**: Phrases that clearly reflect biased or stigmatizing language

## Configuration

Edit the settings below before running the assessment.

In [ ]:
from project_config import (
    BIAS_NOTES_CLEANED_CSV,
    BIAS_NOTES_OUTPUT_DIR,
    BIAS_NOTES_PROMPT_PATH,
)

# =============================================================================
# CONFIGURATION - Edit these settings before running
# =============================================================================

INPUT_CSV = BIAS_NOTES_CLEANED_CSV
OUTPUT_DIR = BIAS_NOTES_OUTPUT_DIR
OUTPUT_PREFIX = "bias_flagged"
PROMPT_PATH = BIAS_NOTES_PROMPT_PATH

# Number of rows to process (set to None to process ALL rows)
N_ROWS_TO_PROCESS = 10
RANDOM_SELECTION = True
RANDOM_SEED = None

CHUNK_CHAR_LIMIT = 2800
PARALLEL_CALLS = 4
GLOBAL_MAX_CALLS = 8
MAX_RETRIES = 5
SLEEP_BASE_SEC = 1.4
TEMPERATURE = 0  # use 1 for GPT-5-family models if required

if not INPUT_CSV:
    raise RuntimeError("Set BIAS_NOTES_CLEANED_CSV in .env before running this notebook.")
if not OUTPUT_DIR:
    raise RuntimeError("Set BIAS_NOTES_OUTPUT_DIR in .env before running this notebook.")
if not PROMPT_PATH:
    raise RuntimeError("Set BIAS_NOTES_PROMPT_PATH in .env before running this notebook.")


## Run Assessment

Execute the cell below to run the bias assessment on your clinical notes.

In [ ]:
# Shared Azure bias-assessment engine
import os
import time

import pandas as pd
from dotenv import load_dotenv
from openai import AzureOpenAI

from bias_pipeline import AzureBiasPipeline, AzureBiasPipelineConfig, generate_output_path, write_output_bundle

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
subscription_key = os.getenv("AZURE_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_API_VERSION")
model_for_api = os.getenv("AZURE_DEPLOYMENT") or os.getenv("AZURE_MODEL_NAME")

if not all([endpoint, subscription_key, api_version, model_for_api]):
    raise RuntimeError(
        "Missing Azure OpenAI configuration. Check AZURE_OPENAI_ENDPOINT, "
        "AZURE_SUBSCRIPTION_KEY, AZURE_API_VERSION, and AZURE_DEPLOYMENT/AZURE_MODEL_NAME."
    )

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

output_csv = globals().get("OUTPUT_CSV")
if not output_csv:
    output_csv = generate_output_path(OUTPUT_DIR, OUTPUT_PREFIX)

df_full = pd.read_csv(INPUT_CSV)
if "note_text" not in df_full.columns:
    raise ValueError("Input CSV must contain a 'note_text' column.")

total_rows = len(df_full)
n_to_process = total_rows if N_ROWS_TO_PROCESS is None else min(N_ROWS_TO_PROCESS, total_rows)

if RANDOM_SELECTION and n_to_process < total_rows:
    if RANDOM_SEED is not None:
        df = df_full.sample(n=n_to_process, random_state=RANDOM_SEED).reset_index(drop=True)
    else:
        df = df_full.sample(n=n_to_process).reset_index(drop=True)
    selection_mode = f"random sample (seed={RANDOM_SEED})" if RANDOM_SEED is not None else "random sample"
else:
    df = df_full.head(n_to_process).reset_index(drop=True)
    selection_mode = "first N rows" if n_to_process < total_rows else "all rows"

pipeline = AzureBiasPipeline(
    client,
    AzureBiasPipelineConfig(
        prompt_path=PROMPT_PATH,
        model_for_api=model_for_api,
        temperature=TEMPERATURE,
        chunk_char_limit=CHUNK_CHAR_LIMIT,
        max_retries=MAX_RETRIES,
        sleep_base_sec=SLEEP_BASE_SEC,
        parallel_calls=PARALLEL_CALLS,
        global_max_calls=GLOBAL_MAX_CALLS,
    ),
)

print(f"Input file: {INPUT_CSV}")
print(f"Total rows in file: {total_rows}")
print(f"Rows to process: {n_to_process} ({selection_mode})")
print(f"Output will be saved to: {output_csv}")
print(f"Model: {model_for_api}")
print(f"PARALLEL_CALLS={PARALLEL_CALLS}")
print("-" * 50)

start_time = time.time()
df = pipeline.process_dataframe(df)
output_paths = write_output_bundle(df, output_csv)
total_time = time.time() - start_time

OUTPUT_CSV = output_paths["reviewer_path"]
ANALYSIS_READY_CSV = output_paths["analysis_path"]
REVIEWER_ADJUDICATION_CSV = output_paths["adjudication_path"]
print("-" * 50)
print(f"Reviewer output: {OUTPUT_CSV}")
print(f"Analysis-ready output: {ANALYSIS_READY_CSV}")
print(f"Reviewer adjudication output: {REVIEWER_ADJUDICATION_CSV}")
print(f"Possible bias flags kept: {int(df['Possible_Bias_Count'].sum())}")
print(f"Likely bias flags kept: {int(df['Likely_Bias_Count'].sum())}")
print(f"Total time: {total_time:.1f}s ({total_time / len(df):.1f}s per note)")


## Preview Results (Optional)

Run the cell below to preview the results.

In [ ]:
# =============================================================================
# RESULTS SUMMARY - Bias Detection Rates by Note Type (Human vs DAX)
# =============================================================================

print("=" * 70)
print("BIAS DETECTION RESULTS SUMMARY")
print("=" * 70)

# Helper function to check if a JSON array string is non-empty
def has_terms(json_str):
    return json_str != "[]" and json_str != ""

# Add helper columns for analysis
df["has_possible"] = df["Possible_Biased_Terms"].apply(has_terms)
df["has_likely"] = df["Likely_Biased_Terms"].apply(has_terms)
df["has_any_bias"] = df["has_possible"] | df["has_likely"]

# Overall statistics
total_notes = len(df)
notes_with_possible = df["has_possible"].sum()
notes_with_likely = df["has_likely"].sum()
notes_with_any = df["has_any_bias"].sum()

print(f"\n{'OVERALL RESULTS':^70}")
print("-" * 70)
print(f"Total notes analyzed: {total_notes}")
print(f"")
print(f"  Notes with POSSIBLE bias:  {notes_with_possible:4d}  ({100*notes_with_possible/total_notes:5.1f}%)")
print(f"  Notes with LIKELY bias:    {notes_with_likely:4d}  ({100*notes_with_likely/total_notes:5.1f}%)")
print(f"  Notes with ANY bias:       {notes_with_any:4d}  ({100*notes_with_any/total_notes:5.1f}%)")

# Check if Dax_or_Human column exists for breakdown
if "Dax_or_Human" in df.columns:
    print(f"\n{'RESULTS BY NOTE TYPE':^70}")
    print("-" * 70)
    
    # Get unique note types
    note_types = df["Dax_or_Human"].unique()
    
    # Create summary table
    summary_data = []
    
    for note_type in sorted(note_types):
        subset = df[df["Dax_or_Human"] == note_type]
        n = len(subset)
        n_possible = subset["has_possible"].sum()
        n_likely = subset["has_likely"].sum()
        n_any = subset["has_any_bias"].sum()
        
        summary_data.append({
            "Note Type": note_type,
            "Total": n,
            "Possible Bias": n_possible,
            "Possible %": f"{100*n_possible/n:.1f}%" if n > 0 else "N/A",
            "Likely Bias": n_likely,
            "Likely %": f"{100*n_likely/n:.1f}%" if n > 0 else "N/A",
            "Any Bias": n_any,
            "Any %": f"{100*n_any/n:.1f}%" if n > 0 else "N/A",
        })
        
        print(f"\n  {note_type.upper()} Notes (n={n}):")
        print(f"    Possible bias: {n_possible:4d}  ({100*n_possible/n:5.1f}%)" if n > 0 else "    Possible bias: N/A")
        print(f"    Likely bias:   {n_likely:4d}  ({100*n_likely/n:5.1f}%)" if n > 0 else "    Likely bias:   N/A")
        print(f"    Any bias:      {n_any:4d}  ({100*n_any/n:5.1f}%)" if n > 0 else "    Any bias:      N/A")
    
    # Display as DataFrame table
    print(f"\n{'COMPARISON TABLE':^70}")
    print("-" * 70)
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)
    
    # Calculate and display difference if both Human and Dax exist
    if "Human" in note_types and "Dax" in note_types:
        human_df = df[df["Dax_or_Human"] == "Human"]
        dax_df = df[df["Dax_or_Human"] == "Dax"]
        
        human_any_rate = 100 * human_df["has_any_bias"].sum() / len(human_df) if len(human_df) > 0 else 0
        dax_any_rate = 100 * dax_df["has_any_bias"].sum() / len(dax_df) if len(dax_df) > 0 else 0
        
        human_likely_rate = 100 * human_df["has_likely"].sum() / len(human_df) if len(human_df) > 0 else 0
        dax_likely_rate = 100 * dax_df["has_likely"].sum() / len(dax_df) if len(dax_df) > 0 else 0
        
        print(f"\n{'HUMAN vs DAX COMPARISON':^70}")
        print("-" * 70)
        print(f"  Any bias rate difference:    {human_any_rate - dax_any_rate:+.1f} percentage points (Human - DAX)")
        print(f"  Likely bias rate difference: {human_likely_rate - dax_likely_rate:+.1f} percentage points (Human - DAX)")
        
else:
    print("\n[INFO] Column 'Dax_or_Human' not found - skipping breakdown by note type.")
    print("       To see Human vs DAX comparison, ensure your input CSV has a 'Dax_or_Human' column.")

# Clean up helper columns before final display
df_display = df.drop(columns=["has_possible", "has_likely", "has_any_bias"], errors="ignore")

print(f"\n{'SAMPLE FLAGGED NOTES':^70}")
print("-" * 70)

# Show notes that have at least one flagged term
flagged = df[df["has_any_bias"]]
print(f"Notes with flagged terms: {len(flagged)} / {len(df)}")

if len(flagged) > 0:
    print("\nFirst few flagged notes:")
    display_cols = ["Likely_Biased_Terms", "Possible_Biased_Terms"]
    if "Dax_or_Human" in df.columns:
        display_cols = ["Dax_or_Human"] + display_cols
    display(flagged[display_cols].head(10))